In [1]:
from pathlib import Path
import sys
from typing import Any
import pandas as pd
from IPython.display import display
from sqlalchemy import text

here = Path.cwd().resolve()
project_root = next((p for p in [here, *here.parents] if (p / "App").is_dir()), None)
if project_root is None:
    raise RuntimeError("Start Jupyter from inside the TechAdmin repository")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from App.db.connection import DB_SCHEMA, engine
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
print(f"Project root: {project_root}")
print(f"Configured schema: {DB_SCHEMA}")

Project root: D:\Projects\TechAdmin
Configured schema: techadmin


In [2]:
def query_df(sql: str, params: dict[str, Any] | None = None) -> pd.DataFrame:
    # Execute a parameterized SELECT statement and return a DataFrame.
    with engine.connect() as connection:
        return pd.read_sql_query(text(sql), connection, params=params or {})

In [3]:
connection_df = query_df("""
SELECT current_database() AS database_name,
       current_user AS database_user,
       current_schema() AS current_schema,
       current_setting('search_path') AS search_path,
       inet_server_addr()::text AS server_address,
       inet_server_port() AS server_port
""")
display(connection_df)

,database_name,database_user,current_schema,search_path,server_address,server_port
0,techadmin_dev,postgres,techadmin,"techadmin,public",::1/128,5432


In [4]:
app_users_df = query_df(f"""
SELECT user_id, entra_object_id, user_principal_name, display_name,
       department, is_active, created_at, updated_at
FROM "{DB_SCHEMA}"."app_users"
ORDER BY display_name
""")
display(app_users_df)

,user_id,entra_object_id,user_principal_name,display_name,department,is_active,created_at,updated_at
0,ef242f27-2b98-43e0-a14d-deacbaaf51e8,ae137d62-060e-4598-9616-60eac23ebb1c,aman.14.gupta@coforge.com,Aman Gupta - GNoida,IT,True,2026-09-09 09:48:23.965802+00:00,2026-09-09 09:48:23.965802+00:00
1,7e37ee55-9d5b-4b9e-b01d-0f26f5dff118,94dde518-4345-4665-a8d3-860c852e669d,aman.3.mishra@coforge.com,Aman Mishra,IT,True,2026-09-09 09:48:23.908187+00:00,2026-09-09 09:48:23.908187+00:00
2,5ff2fa29-297b-415d-a363-c7dce26e5168,bffa5400-3970-45f1-b022-13c39d0c0bd7,amit.bhagat@coforge.com,Amit Bhagat,IT,True,2026-09-09 09:48:23.943905+00:00,2026-09-09 09:48:23.943905+00:00
3,f1964853-bede-44a8-846d-857356ad28fe,d7aac575-8f3e-4fd3-a821-db94e4afcb55,jitender.chauhan@coforge.com,Jitender Chauhan - Global IT,IT,True,2026-09-09 10:22:25.036313+00:00,2026-09-09 10:22:25.036325+00:00
4,897139ee-531c-41f9-99ba-ab552e5e36bf,09c5f133-4294-4ef7-b1df-e958fe339f32,roshan.sah@coforge.com,Roshan Sah,IT,True,2026-09-09 09:48:23.952328+00:00,2026-09-09 09:48:23.952328+00:00
5,1375dc37-9516-4255-b450-993a29c1fcfd,ff614b3d-1fdb-453d-b3d9-809ec9bc26fb,shreesanyog.rath@coforge.com,Shreesanyog Rath,IT,True,2026-09-09 09:48:23.959351+00:00,2026-09-09 09:48:23.959351+00:00
6,842f669d-5bd3-427f-be4f-b6aeb35dd560,None,techadmintestuser@coforge.com,TechAdminTestUser,IT,True,2026-09-16 10:21:37.135335+00:00,2026-09-16 10:21:37.135340+00:00


In [12]:
operation_req_df = query_df(f"""
SELECT *
FROM "{DB_SCHEMA}"."operation_requests"
ORDER BY requested_at desc
""")
display(operation_req_df)

,request_id,requested_by,operation_id,source_channel,original_request,target_type,target_reference,target_object_id,request_parameters,intent_confidence,status,requested_at,completed_at
0,fab8c78a-0378-5832-8567-2d6215389b96,None,None,WEB,Get user details for amit.bhagat,NaN,NaN,None,None,0.00,REJECTED,2026-09-18 08:02:55.407505+00:00,2026-09-18 08:02:55.435728+00:00
1,76d40443-823a-544a-b0f7-baedf75b7c90,None,f97d0386-a278-4a37-86e0-f39e237b0749,WEB,Get user details for roshan.sah@coforge.com,USER,ro********@coforge.com,None,"{'email': 'roshan.sah@coforge.com', 'username': 'roshan.sah'}",0.99,SUCCEEDED,2026-09-18 07:23:10.401793+00:00,2026-09-18 07:23:30.357825+00:00
2,47732640-c050-51c1-a8ae-df075e5cc0de,None,None,WEB,test query to check db operation,NaN,NaN,None,None,0.00,REJECTED,2026-09-18 07:19:29.607002+00:00,2026-09-18 07:19:29.633842+00:00
3,ba16f761-7288-5ed1-80d5-fa8db182363d,5ff2fa29-297b-415d-a363-c7dce26e5168,None,WEB,Can you give me details of all users,NaN,NaN,None,None,0.00,REJECTED,2026-09-18 06:56:04.573057+00:00,2026-09-18 06:56:04.591426+00:00
4,7e4cb576-22cb-5f22-bd3d-92c9abbe880b,5ff2fa29-297b-415d-a363-c7dce26e5168,f97d0386-a278-4a37-86e0-f39e237b0749,WEB,Get user details for ravi.dharavath@coforge.com,USER,ra************@coforge.com,None,"{'email': 'ravi.dharavath@coforge.com', 'username': 'ravi.dharavath'}",0.99,SUCCEEDED,2026-09-18 06:43:36.244188+00:00,2026-09-18 06:43:48.973978+00:00
5,0ee0af0a-d163-5f7c-9bef-7fc7217b4b79,5ff2fa29-297b-415d-a363-c7dce26e5168,f97d0386-a278-4a37-86e0-f39e237b0749,WEB,Get user details for ritesh.saluja@coforge.com,USER,ri***********@coforge.com,None,"{'email': 'ritesh.saluja@coforge.com', 'username': 'ritesh.saluja'}",0.99,SUCCEEDED,2026-09-18 06:42:22.317207+00:00,2026-09-18 06:42:34.861790+00:00
6,4b61e2ee-4d94-5ac4-9542-ff293d6d7008,5ff2fa29-297b-415d-a363-c7dce26e5168,f97d0386-a278-4a37-86e0-f39e237b0749,WEB,Get user details for derhant@coforge.com,USER,de*****@coforge.com,None,"{'email': 'derhant@coforge.com', 'username': 'derhant'}",0.99,SUCCEEDED,2026-09-18 06:37:54.125836+00:00,2026-09-18 06:38:16.021797+00:00
7,7a43f9e7-e3de-5b65-bb38-1a79541fd3cc,5ff2fa29-297b-415d-a363-c7dce26e5168,f97d0386-a278-4a37-86e0-f39e237b0749,WEB,Get user details for amit.bhagat@coforge.com,USER,am*********@coforge.com,None,"{'email': 'amit.bhagat@coforge.com', 'username': 'amit.bhagat'}",0.99,SUCCEEDED,2026-09-17 10:24:57.349566+00:00,2026-09-17 10:25:08.914208+00:00
8,3c5ddb02-25ba-57ec-b0b7-aa09981512e9,5ff2fa29-297b-415d-a363-c7dce26e5168,None,WEB,Delete all users,NaN,NaN,None,None,0.00,REJECTED,2026-09-17 09:47:45.714258+00:00,2026-09-17 09:47:45.725296+00:00
9,b2b2c6c9-7a94-5253-a2fa-334e817101fc,5ff2fa29-297b-415d-a363-c7dce26e5168,f97d0386-a278-4a37-86e0-f39e237b0749,WEB,Get User Details for sarbojeet.mondal@coforge.com,USER,sa**************@coforge.com,None,"{'email': 'sarbojeet.mondal@coforge.com', 'username': 'sarbojeet.mondal'}",0.99,SUCCEEDED,2026-09-17 07:22:32.853030+00:00,2026-09-17 07:22:53.361527+00:00


In [9]:
operation_exe_df = query_df(f"""
SELECT *
FROM "{DB_SCHEMA}"."operation_executions"
ORDER BY started_at desc
""")
display(operation_exe_df)

,execution_id,request_id,execution_type,executor_name,executor_version,executor_host,started_at,finished_at,execution_status,http_status_code,process_exit_code,external_reference_id,duration_ms,result_summary,error_code,error_message,retry_count
0,4b1287d9-f206-40fb-b57a-7e871dfd0e5d,76d40443-823a-544a-b0f7-baedf75b7c90,API,get_user_details,None,IN-TZ1-AIOPS1,2026-09-18 07:23:30.370656+00:00,2026-09-18 07:23:30.370662+00:00,SUCCEEDED,None,None,corr_b59d33b0443e44f582253b6550e141fc,0,"User details retrieved through Microsoft Graph API. | fields=backend,user",None,None,0
1,9e769d2c-2327-4f57-8b14-8ce1e74f4993,7e4cb576-22cb-5f22-bd3d-92c9abbe880b,API,get_user_details,None,IN-TZ1-AIOPS1,2026-09-18 06:43:48.980911+00:00,2026-09-18 06:43:48.980917+00:00,SUCCEEDED,None,None,corr_62439d3c4b4f4042abb2bb75ccdb32d8,0,"User details retrieved through Microsoft Graph API. | fields=backend,user",None,None,0
2,94a9b040-ae84-486d-baf1-6b4e8b53bacf,0ee0af0a-d163-5f7c-9bef-7fc7217b4b79,API,get_user_details,None,IN-TZ1-AIOPS1,2026-09-18 06:42:34.871314+00:00,2026-09-18 06:42:34.871320+00:00,SUCCEEDED,None,None,corr_cef4c83f11ae4b3f8475383e4c2bd29d,0,"User details retrieved through Microsoft Graph API. | fields=backend,user",None,None,0
3,09337950-a918-4010-8843-b271f9a00742,4b61e2ee-4d94-5ac4-9542-ff293d6d7008,API,get_user_details,None,IN-TZ1-AIOPS1,2026-09-18 06:38:16.033486+00:00,2026-09-18 06:38:16.033490+00:00,SUCCEEDED,None,None,corr_97117b2b1dfe42fcb9eb93203331d979,0,"User details retrieved through Microsoft Graph API. | fields=backend,user",None,None,0
4,be6b8f5a-b146-4c70-9eac-ea0792701786,7a43f9e7-e3de-5b65-bb38-1a79541fd3cc,API,get_user_details,None,IN-TZ1-AIOPS1,2026-09-17 10:25:08.920287+00:00,2026-09-17 10:25:08.920290+00:00,SUCCEEDED,None,None,corr_3524b00cc3254f9e93a572091e002d88,0,"User details retrieved through Microsoft Graph API. | fields=backend,user",None,None,0
